# Experiment 01 — Resource-aware EfficientNet-B0 flood segmentation

This is the first real experiment for **Small Models, Honest Maps**.

- **Model:** UNet + EfficientNet-B0
- **Input:** Sentinel-1 VV, VH, stabilized ratio
- **Training regions:** Nebraska, North Alabama, Bangladesh, Red River North
- **Held-out region:** Florence
- **Training:** 60 epochs, physical batch 8, gradient accumulation 4 (effective batch 32), AMP, cosine schedule

The ETCI data workflow is intentionally persistent: the required VV/VH/flood-label subset is downloaded to Google Drive once, then copied to Colab's local disk for faster training.

## 0. GPU check

In Colab choose **Runtime → Change runtime type → GPU** before running this notebook. The GPU will remain mostly idle during data download/copy; that is normal.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

subprocess.run(["nvidia-smi"], check=False)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required for the full experiment. Enable a Colab GPU and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

## 1. Clone `main` and install the project

In [ ]:
REPO = "https://github.com/nazizahed/Uncertainty-Aware-Flood-Segmentation-from-Sentinel-1-for-Near-Real-Time-Applications.git"
BRANCH = "main"
WORKDIR = "/content/sar-flood-uq"

def run_cmd(args, cwd=None, env=None):
    print("$", " ".join(map(str, args)), flush=True)
    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    subprocess.run(list(map(str, args)), cwd=cwd, env=merged_env, check=True)

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
run_cmd(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, WORKDIR])
os.chdir(WORKDIR)
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Repository ready at", WORKDIR)
print("CUDA still available after install:", torch.cuda.is_available())

## 2. Mount Google Drive

Both experiment outputs and the required ETCI data subset are kept permanently in Drive. The first session downloads the dataset subset; later sessions reuse it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/sar-flood-uq"
DRIVE_DATA = f"{DRIVE_ROOT}/data/etci"
EXP_NAME = "lightweight_unet_efficientnetb0"
DRIVE_RUN = f"{DRIVE_ROOT}/runs/{EXP_NAME}"
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_RUN, exist_ok=True)
print("Persistent ETCI directory:", DRIVE_DATA)
print("Persistent run directory:", DRIVE_RUN)

## 3. Repository checks

Do not spend GPU time if these checks fail.

In [ ]:
run_cmd([sys.executable, "scripts/validate_repository.py"], cwd=WORKDIR)
run_cmd([sys.executable, "-m", "pytest", "-q"], cwd=WORKDIR)
print("Repository checks passed.")

## 4. Get only the ETCI files used by Experiment 01

The model uses only **VV**, **VH**, and **flood_label** PNGs. We deliberately skip `water_body_label` and other unused files.

**First run:** download this required subset to Google Drive. This may still take a while because ETCI contains many small files, but the completed files remain in Drive even if the Colab session ends.

**Later runs:** the download is skipped and the persistent Drive copy is reused.

After that, the subset is copied to `/content` because training from Colab's local disk is much faster than reading thousands of small images directly from Drive.

In [ ]:
HF_REPO = "blanchon/ETCI-2021-Flood-Detection"
DATA_ROOT = f"{WORKDIR}/data/etci"
REQUIRED_PATTERNS = [
    "data/*/*/tiles/vv/*.png",
    "data/*/*/tiles/vh/*.png",
    "data/*/*/tiles/flood_label/*.png",
]

def etci_subset_ready(root):
    root = Path(root)
    required_splits = [root / "data/train", root / "data/test", root / "data/test_internal"]
    if not all(path.is_dir() for path in required_splits):
        return False
    for folder_name in ("vv", "vh", "flood_label"):
        if not any(root.glob(f"data/*/*/tiles/{folder_name}/*.png")):
            return False
    return True

if not etci_subset_ready(DRIVE_DATA):
    print("Required ETCI subset is not complete in Drive; downloading/resuming it now.")
    cmd = [
        "hf", "download", HF_REPO,
        "--repo-type", "dataset",
        "--local-dir", DRIVE_DATA,
    ]
    for pattern in REQUIRED_PATTERNS:
        cmd.extend(["--include", pattern])
    run_cmd(cmd, env={"HF_HUB_DISABLE_XET": "1"})
else:
    print("Persistent ETCI subset already exists in Drive; skipping internet download.")

if not etci_subset_ready(DRIVE_DATA):
    raise RuntimeError("ETCI subset is still incomplete after the download step.")

print("Copying persistent ETCI subset from Drive to Colab local disk for fast training...")
os.makedirs(DATA_ROOT, exist_ok=True)
run_cmd(["rsync", "-a", "--delete", f"{DRIVE_DATA}/", f"{DATA_ROOT}/"])

if not etci_subset_ready(DATA_ROOT):
    raise RuntimeError("Local ETCI copy is incomplete.")

print("ETCI subset ready for training at:", DATA_ROOT)

## 4b. Inspect the train/Florence split

In [ ]:
import yaml

sys.path.insert(0, str(Path(WORKDIR) / "src"))
from sarflood.data.dataset import ETCIFloodDataset

with open(Path(WORKDIR) / "configs/lightweight_unet_b0.yaml") as f:
    cfg = yaml.safe_load(f)

train_ds = ETCIFloodDataset(
    cfg["data"]["root"],
    cfg["data"]["regions"],
    cfg["data"]["bands"],
    rotation_aug=cfg["data"]["rotation_aug"],
    image_size=cfg["data"]["image_size"],
    ratio_clip=cfg["data"]["ratio_clip"],
)
val_ds = ETCIFloodDataset(
    cfg["data"]["root"],
    cfg["data"]["val_regions"],
    cfg["data"]["bands"],
    rotation_aug=False,
    image_size=cfg["data"]["image_size"],
    ratio_clip=cfg["data"]["ratio_clip"],
)
print(f"Base training tiles: {len(train_ds.records):,}")
print(f"Training samples after rotations: {len(train_ds):,}")
print(f"Florence validation tiles: {len(val_ds):,}")
sample = train_ds[0]
print("Input shape:", tuple(sample["image"].shape))
print("Mask shape:", tuple(sample["mask"].shape))
assert sample["image"].shape[0] == 3
assert sample["mask"].shape[0] == 1

## 5. Confirm the exact Experiment 01 configuration

In [ ]:
from pprint import pprint

pprint(cfg)
physical_batch = cfg["training"]["batch_size"]
accum = cfg["training"].get("gradient_accumulation_steps", 1)
print("Physical batch:", physical_batch)
print("Gradient accumulation:", accum)
print("Effective batch:", physical_batch * accum)
assert cfg["model"]["arch"] == "unet"
assert cfg["model"]["encoder"] == "efficientnet-b0"
assert physical_batch == 8
assert accum == 4

## 6. One-batch GPU sanity check

This is the point to stop if memory usage is unexpectedly high.

In [ ]:
from sarflood.data.dataset import build_dataloaders
from sarflood.models.build import build_model
from sarflood.training.losses import BCEDiceLoss

device = "cuda"
train_loader, _ = build_dataloaders(cfg)
model = build_model(cfg["model"], in_channels=len(cfg["data"]["bands"])).to(device)
loss_fn = BCEDiceLoss()
batch = next(iter(train_loader))
img = batch["image"].to(device)
mask = batch["mask"].to(device)
model.train()
torch.cuda.reset_peak_memory_stats()
with torch.cuda.amp.autocast(enabled=True):
    logits = model(img)
    loss = loss_fn(logits, mask) / accum
loss.backward()
print("Scaled sanity loss:", float(loss.detach().cpu()))
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
del model, img, mask, logits, loss, batch, train_loader
torch.cuda.empty_cache()
print("One-batch sanity check passed.")

## 7. Full 60-epoch training

The run writes `best.pt`, `last.pt`, `log.csv`, and `config.yaml`. The cell copies the final artifacts to Drive when training exits.

In [ ]:
RUN_DIR = f"{WORKDIR}/runs/{EXP_NAME}"
run_cmd([sys.executable, "scripts/train.py", "--config", "configs/lightweight_unet_b0.yaml"], cwd=WORKDIR)
os.makedirs(DRIVE_RUN, exist_ok=True)
for name in ["best.pt", "last.pt", "log.csv", "config.yaml"]:
    src = os.path.join(RUN_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RUN, name))
print("Training artifacts copied to:", DRIVE_RUN)

## 8. Inspect convergence

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv(os.path.join(RUN_DIR, "log.csv"))
display(hist.tail(10))
best_idx = hist["eval_score"].idxmax()
print("Best epoch:")
display(hist.loc[[best_idx]])

plt.figure(figsize=(8, 5))
plt.plot(hist["epoch"], hist["train_loss"], label="train loss")
plt.plot(hist["epoch"], hist["val_loss"], label="Florence loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(hist["epoch"], hist["f1"], label="F1")
plt.plot(hist["epoch"], hist["iou"], label="pooled IoU")
plt.plot(hist["epoch"], hist["miou_tiles"], label="mean tile IoU")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True)
plt.legend()
plt.show()

## 9. Deterministic Florence evaluation

This is the first primary result: segmentation quality, calibration, and one-pass uncertainty baselines.

In [ ]:
RESULT_DIR = f"{RUN_DIR}/results"
os.makedirs(RESULT_DIR, exist_ok=True)
det_path = f"{RESULT_DIR}/florence_deterministic.json"
run_cmd([
    sys.executable, "scripts/evaluate.py",
    "--checkpoint", f"{RUN_DIR}/best.pt",
    "--regions", "florence",
    "--out", det_path,
], cwd=WORKDIR)
shutil.copy2(det_path, os.path.join(DRIVE_RUN, "florence_deterministic.json"))

In [ ]:
import json
import pprint

with open(det_path) as f:
    det = json.load(f)
print("Segmentation metrics")
pprint.pp(det["metrics"])
print("\nCalibration")
pprint.pp(det["calibration"])
print("\nDeterministic selective prediction")
pprint.pp({
    name: {"aurc": values["aurc"], "sparsification_error": values["sparsification_error"]}
    for name, values in det["selective_prediction"].items()
})

## 10. MC-dropout with 5 passes

We start at 5 stochastic passes. Do **not** automatically escalate to 10 or 20; first compare the reliability gain with the extra inference cost.

In [ ]:
mc5_path = f"{RESULT_DIR}/florence_mc5.json"
run_cmd([
    sys.executable, "scripts/evaluate.py",
    "--checkpoint", f"{RUN_DIR}/best.pt",
    "--regions", "florence",
    "--mc-passes", "5",
    "--out", mc5_path,
], cwd=WORKDIR)
shutil.copy2(mc5_path, os.path.join(DRIVE_RUN, "florence_mc5.json"))

with open(mc5_path) as f:
    mc5 = json.load(f)
pprint.pp({
    name: {"aurc": values["aurc"], "sparsification_error": values["sparsification_error"]}
    for name, values in mc5["selective_prediction"].items()
})

## 11. Decision gate

Record the following before spending more GPU time:

1. best Florence F1, pooled IoU, and mean-tile IoU;
2. overall/flood/non-flood calibration;
3. deterministic entropy/confidence AURC;
4. MC-5 predictive entropy, mutual-information, and variance AURC;
5. training time and peak GPU memory.

The next experiment is chosen from these results, not automatically.